In [0]:
%sql
CREATE VOLUME IF NOT EXISTS uc.urise.urise_etc;



In [0]:
# Bypass CSV - langsung baca dari tabel existing
df = spark.table("uc.default.stockpile_transaction")
print(f"Total rows: {df.count()}")

df.write.mode("overwrite").saveAsTable('uc.urise.stockpile_bronze')

In [0]:
# Schema sudah ada di uc.urise, skip create schema


In [0]:
df_sp_bronze = spark.table("uc.urise.stockpile_bronze")

In [0]:
import pandas as pd
df = df_sp_bronze.toPandas()
# 1
# Konversi room_in menjadi datetime
df["room_in"] = pd.to_datetime(df["room_in"])

# 2. Urutkan data berdasarkan unit dan waktu masuk
df = df.sort_values(by=['no_lambung', 'room_in'])

# 3. Hitung selisih waktu dengan baris sebelumnya (dalam menit)
df['selisih_menit'] = df.groupby('no_lambung')['room_in'].diff().dt.total_seconds() / 60

# 4. Definisikan kriteria duplikat (jarak <= 20 menit DAN room_out-nya kosong)
is_duplicate = (df['selisih_menit'] <= 20) & (df['room_out'].isna())

# 5. Hapus data duplikat
df_clean = df[~is_duplicate].drop(columns=['selisih_menit'])

# Ambil jam
df_clean["hour"] = df_clean["room_in"].dt.hour

# Tentukan tanggal operasional
# Jika sebelum jam 06:00 maka dianggap hari sebelumnya
operational_datetime = df_clean["room_in"].where(
    df_clean["hour"] >= 6,
    df_clean["room_in"] - pd.Timedelta(days=1)
)

df_clean["op_date"] = operational_datetime.dt.date


# Tentukan shift
df_clean["shift"] = df_clean["hour"].apply(
    lambda h: "DS" if 6 <= h < 18 else "NS"
)
dfs = spark.createDataFrame(df_clean)
dfs.write.mode("overwrite").saveAsTable('uc.urise.stockpile_silver')

In [0]:
# Schema sudah ada di uc.urise, skip create schema


In [0]:
df_sp_silver = spark.table("uc.urise.stockpile_silver")

In [0]:
import pandas as pd

df = df_sp_silver.toPandas()

# Ubah tipe data kolom
df["room_in"] = pd.to_datetime(df["room_in"])
df["room_out"] = pd.to_datetime(df["room_out"])
# Pastikan kolom string tidak berisi spasi
df["location"] = df["location"].fillna("").astype(str).str.strip()
df["location_out"] = df["location_out"].fillna("").astype(str).str.strip()

# Pastikan tonase numerik
df["tonase"] = pd.to_numeric(df["tonase"], errors="coerce").fillna(0)

# Flag untuk agregasi
df["inbound"] = (df["location"].str.endswith('IN') | df["location"].str.contains("IN")) & (df["tonase"] > 0)
df["outbound"] = df["location_out"].str.endswith('OUT') | df["location"].str.contains("OUT")
df["aktif_geofence"] = (
    (df["location"] != "") &
    (df["location_out"] == "")
)

# Pembuatan kolom kustom untuk jumlah tonase spesifik kondisi
# (Menghasilkan nilai tonase jika True, dan 0 jika False)
df["tonase_inbound_raw"] = df["tonase"].where(df["inbound"], 0)
df["tonase_outbound_raw"] = df["tonase"].where(df["outbound"], 0)

# Agregasi
gold = (
    df.groupby(["op_date", "shift", "hour"], as_index=False)
    .agg(
        jumlah_inbound=("inbound", "sum"),
        jumlah_outbound=("outbound", "sum"),
        jumlah_aktif_geofence=("aktif_geofence", "sum"),
        jumlah_tonase=("tonase", "sum"),
        tonase_inbound=("tonase_inbound_raw", "sum"),
        tonase_outbound=("tonase_outbound_raw", "sum"),
    )
    .sort_values(["op_date", "shift", "hour"])
)

# # Simpan hasil
# gold.to_csv(output_file, index=False)

# print(f"Agregasi selesai. Hasil disimpan ke {output_file}")



dfg = spark.createDataFrame(gold)
dfg.write.mode("overwrite").saveAsTable('uc.urise.stockpile_gold')

In [0]:
%sql
SELECT
  op_date,
  shift,
  hour,
  jumlah_inbound,
  jumlah_outbound,
  jumlah_aktif_geofence,
  jumlah_tonase,
  jumlah_tonase_inbound,
  jumlah_tonase_outbound
FROM
  uc.urise.stockpile_gold
ORDER BY
  op_date,
  shift,
  hour